# CS604 Colab launcher

Sign in with **thv.le.2025@mitb.smu.edu.sg** (the Colab Pro account).

In the Colab website: Runtime → Change runtime type → GPU.

In Cursor: open this notebook, Select Kernel → Colab → New Colab Server → GPU, and pick that Google account when the browser prompt appears.

This notebook only launches `scripts/train.py`. Edit the training code in `src/` from Cursor, commit and push to `prototype-baseline`, then set `COMMIT` below to that full SHA before a real run.

In [ ]:
import subprocess
subprocess.run(["nvidia-smi"], check=False)
try:
    import google.colab  # noqa: F401
except ImportError:
    raise SystemExit("This notebook needs a Colab kernel. Locally, run scripts/train.py with configs/local.yaml.")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/levanjan93/cs604-group-project.git"
# Pin to the exact commit that passed local tests (prototype-baseline branch). Not "main" for real runs.
COMMIT = "302d0ef07d55aeb877f995ea36e136b2f282a90e"
code = Path("/content/cs604")

if not (code / ".git").exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(code)])
os.chdir(code)
subprocess.check_call(["git", "fetch", "origin"])
subprocess.check_call(["git", "checkout", "--detach", COMMIT])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"
])
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"], text=True).strip()
if dirty:
    raise SystemExit(f"Checkout is dirty; aborting.\n{dirty}")
print("code ready", code, "commit", commit)

In [ ]:
import hashlib, shutil
from pathlib import Path

EXPECTED = {
    "class_prototypes.pt": "c4c90ef9443c1430c519c4496829117563e21c796037f764a89c5ab1ea11b68f",
    "internal_val.txt": "19a10c37a775559f4eee516215ad10c4b906ee50d66f7cbd95bc5ec88f2f2f69",
}
DRIVE_INPUTS = Path("/content/drive/MyDrive/cs604/inputs")
LOCAL_INPUTS = Path("/content/cs604-inputs")
LOCAL_INPUTS.mkdir(parents=True, exist_ok=True)

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

for name, expected in EXPECTED.items():
    source = DRIVE_INPUTS / name
    if not source.is_file():
        raise SystemExit(f"Missing Drive input: {source}")
    if sha256(source) != expected:
        raise SystemExit(f"Drive hash mismatch for {name}")
    dest = LOCAL_INPUTS / name
    shutil.copy2(source, dest)
    if sha256(dest) != expected:
        raise SystemExit(f"Local copy hash mismatch for {name}")
    print(name, "ok", expected[:12], "...")

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "scripts/stage_data.py",
    "--local-dir", "/content/cs604-course-data",
    "--drive-dir", "/content/drive/MyDrive/cs604/data",
])

In [ ]:
import subprocess, sys

# Run once in a fresh runtime before the 40-image job. Uses its own Drive output dir.
subprocess.check_call([sys.executable, "scripts/colab_checkpoint_smoke.py"])

In [ ]:
import subprocess, sys
from pathlib import Path

# Rank-8 LoRA 40-image/class finalist. Use a new OUTPUT path for smoke tests.
CONFIG = "configs/experiments/prototype_lora_r8_fp32_40.yaml"
OUTPUT = "/content/drive/MyDrive/cs604/outputs/prototype-lora-r8-fp32-40"
RESUME = None  # e.g. "/content/drive/.../checkpoints/resume_step_001000.pt"

command = [
    sys.executable, "scripts/train.py",
    "--config", CONFIG,
    "--data-root", "/content/cs604-course-data",
    "--output-dir", OUTPUT,
]
if RESUME:
    command.extend(["--resume", RESUME])
print("running", " ".join(command))
subprocess.check_call(command)
print("artifacts", OUTPUT)